<a href="https://colab.research.google.com/github/Rakhayeva/multilingual-persuasion-detection/blob/main/01_data_loading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Modules**

# Notebook 01: Data Loading

This notebook loads and parses all datasets used in this project.

## Datasets
- **PTC / SemEval-2020 Task 11** (English): political news articles
  annotated with persuasion techniques at the span level.
  Source: [Zenodo](https://zenodo.org/records/3952415)
- **baiangali/fake_news** (Kazakh + Russian): news articles annotated
  as fake or real. Used as a proxy task for cross-lingual transfer
  experiments, as no persuasion-annotated corpus exists for either
  language in open access.

In [1]:
!pip install shap -q
!pip install langdetect -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 33.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [19]:
!pip install langdetect -q

**Libraries**

In [2]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Project base path
BASE = '/content/drive/MyDrive/NU/SEDS/multilingual-persuasion-detection'

Mounted at /content/drive


In [21]:
# Imports
import json
import glob
import tarfile, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from langdetect import detect, LangDetectException

## SemEval-2020 Task 11 dataset

In [4]:
datasets_path = f'{BASE}/data/raw/datasets'

# Let's look at what's inside the folder with tags
labels_dir = f'{datasets_path}/train-labels-task1-span-identification'
sample_labels = os.listdir(labels_dir)[:5]
print("Examples of label files:")
for f in sample_labels:
    print(f"  {f}")

# Let's look at the contents of one tag file
with open(f'{labels_dir}/{sample_labels[0]}', 'r') as f:
    print(f"\Content {sample_labels[0]}:")
    print(f.read()[:300])

Examples of label files:
  article111111136.task1-SI.labels
  article766632016.task1-SI.labels
  article786344683.task1-SI.labels
  article790667730.task1-SI.labels
  article770156851.task1-SI.labels
\Content article111111136.task1-SI.labels:
111111136	1243	1254
111111136	1874	2026



<>:12: SyntaxWarning: invalid escape sequence '\C'
<>:12: SyntaxWarning: invalid escape sequence '\C'
/tmp/ipykernel_393/2430908769.py:12: SyntaxWarning: invalid escape sequence '\C'
  print(f"\Content {sample_labels[0]}:")


Each line is a span of text containing a persuasive technique. If the file is not empty, the article is manipulative (has_persuasion = 1). If the file is empty, it is 0.

In [8]:
datasets_path = f'{BASE}/data/raw/datasets'
articles_dir  = f'{datasets_path}/train-articles'
labels_dir    = f'{datasets_path}/train-labels-task1-span-identification'

records = []
article_files = glob.glob(f'{articles_dir}/*.txt')
print(f"Articles found: {len(article_files)}")

for art_path in article_files:
    article_id = os.path.basename(art_path).replace('.txt', '')

    # Read the text
    with open(art_path, 'r', encoding='utf-8') as f:
        text = f.read().strip()

    # Looking for a label file
    label_path = f'{labels_dir}/{article_id}.task1-SI.labels'

    if os.path.exists(label_path):
        with open(label_path, 'r', encoding='utf-8') as f:
            content = f.read().strip()
        has_persuasion = 1 if content else 0
    else:
        has_persuasion = None

    if text and has_persuasion is not None:
        records.append({
            'article_id': article_id,
            'text': text,
            'has_persuasion': has_persuasion,
            'language': 'en',
            'source': 'ptc_semeval2020'
        })

df_en = pd.DataFrame(records)
print(f"Articles loaded: {len(df_en)}")
print(df_en['has_persuasion'].value_counts())
print(f"\nExample text (first 200 characters):")
print(df_en.iloc[0]['text'][:200])

Articles found: 371
Articles loaded: 371
has_persuasion
1    357
0     14
Name: count, dtype: int64

Example text (first 200 characters):
'Oumuamua: Space Cigar Is Still Spinning From Mysterious Violent Collision

The solar system’s strange cigar-shaped visitor 'Oumuamua—Hawaiian for “scout” or “messenger”—is tumbling chaotically as the


371 articles, 357 of which were manipulative and only 14 were non-manipulative—a significant imbalance (96% vs. 4%). This is a characteristic of the PTC dataset: it was compiled from deliberately propaganda sources, so almost all the articles contained persuasive techniques.

The dataset contains a dev-articles folder, but there are no labels for task1-SI (only for task2-TC). Let's check what's there:

In [9]:
dev_articles_dir = f'{datasets_path}/dev-articles'
dev_files = glob.glob(f'{dev_articles_dir}/*.txt')
print(f"Dev of articles: {len(dev_files)}")

# Checking if there are tags for dev
dev_labels_candidates = [
    f'{datasets_path}/dev-labels-task1-span-identification',
    f'{datasets_path}/dev-task1-SI.labels',
]
for path in dev_labels_candidates:
    print(f"Exists {path}: {os.path.exists(path)}")

# Let's look at everything that's in datasets/
print("\nAll files and folders in datasets/:")
for item in sorted(os.listdir(datasets_path)):
    print(f"  {item}")

Dev of articles: 75
Exists /content/drive/MyDrive/NU/SEDS/multilingual-persuasion-detection/data/raw/datasets/dev-labels-task1-span-identification: False
Exists /content/drive/MyDrive/NU/SEDS/multilingual-persuasion-detection/data/raw/datasets/dev-task1-SI.labels: False

All files and folders in datasets/:
  README.md
  dev-articles
  dev-task-TC-template.out
  train-articles
  train-labels-task1-span-identification
  train-labels-task2-technique-classification
  train-task1-SI.labels
  train-task2-TC.labels


There are no task1 labels for dev-articles—only for task2 (technique classification). Let's look at train-task1-SI.labels—it's a summary file of all the labels:

In [10]:
# Let's look at the summary file of tags
with open(f'{datasets_path}/train-task1-SI.labels', 'r') as f:
    content = f.read()

lines = content.strip().split('\n')
print(f"Total lines: {len(lines)}")
print("First 5 lines:")
for line in lines[:5]:
    print(f"  {repr(line)}")

Total lines: 5468
First 5 lines:
  '111111111\t265\t323'
  '111111111\t1795\t1935'
  '111111111\t149\t157'
  '111111111\t1069\t1091'
  '111111111\t1334\t1462'


The 357/14 imbalance is a genuine feature of the dataset, not a bug. This can be addressed in two ways simultaneously:
1. class_weight='balanced' — the model automatically compensates for the imbalance.
2. Let's reformulate the problem: instead of the entire article, we'll cut it into paragraphs.
Slicing it into paragraphs gives us many more examples of class 0, because even in a manipulated article, most paragraphs are neutral, and only a few contain persuasive spans.
Each article is split into paragraphs, and each paragraph is assigned a binary label based on character-level span overlap: if any annotated persuasion span from the .task1-SI.labels file intersects with the paragraph's position in the original text,
the paragraph is labeled 1 (manipulative), otherwise 0. This transforms the article-level task into a paragraph-level classification problem with a much more balanced class distribution.

In [11]:
def load_ptc_paragraphs(articles_dir, labels_dir):
    """
    Splits articles into paragraphs and marks each paragraph:
    1 = contains a persuasion span, 0 = does not.
    """
    records = []
    article_files = glob.glob(f'{articles_dir}/*.txt')
    print(f"Articles: {len(article_files)}")

    for art_path in article_files:
        article_id = os.path.basename(art_path).replace('.txt', '')

        with open(art_path, 'r', encoding='utf-8') as f:
            text = f.read()

        # Read persuasion spans for this article
        label_path = f'{labels_dir}/{article_id}.task1-SI.labels'
        persuasion_spans = []
        if os.path.exists(label_path):
            with open(label_path, 'r', encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) == 3:
                        _, start, end = parts
                        persuasion_spans.append((int(start), int(end)))

        # Split into paragraphs
        # Find the position of each paragraph in the text
        paragraphs = text.split('\n\n')
        char_pos = 0

        for para in paragraphs:
            para = para.strip()
            if len(para) < 20:  # skip too short ones
                char_pos += len(para) + 2
                continue

            para_start = text.find(para, char_pos)
            para_end = para_start + len(para)

            # Check for intersections Is there a paragraph with a persuasion span
            has_persuasion = 0
            for span_start, span_end in persuasion_spans:
                # Intersection: span starts before the end of the paragraph
                # И And ends after the start of the paragraph
                if span_start < para_end and span_end > para_start:
                    has_persuasion = 1
                    break

            records.append({
                'article_id': article_id,
                'text': para,
                'has_persuasion': has_persuasion,
                'language': 'en',
                'source': 'ptc_semeval2020'
            })

            char_pos = para_end

    return pd.DataFrame(records)

# Loading
df_en = load_ptc_paragraphs(
    articles_dir=f'{datasets_path}/train-articles',
    labels_dir=f'{datasets_path}/train-labels-task1-span-identification'
)

print(f"\nTotal paragraphs: {len(df_en)}")
print("Class balance:")
print(df_en['has_persuasion'].value_counts())
print(f"\nPercentage of manipulative: "
      f"{df_en['has_persuasion'].mean()*100:.1f}%")
print(f"\nExample of a manipulative paragraph:")
print(df_en[df_en['has_persuasion']==1].iloc[0]['text'][:300])
print(f"\nExample of a neutral paragraph:")
print(df_en[df_en['has_persuasion']==0].iloc[0]['text'][:300])

Статей: 371

Total paragraphs: 1147
Class balance:
has_persuasion
1    644
0    503
Name: count, dtype: int64

Percentage of manipulative: 56.1%

Example of a manipulative paragraph:
The solar system’s strange cigar-shaped visitor 'Oumuamua—Hawaiian for “scout” or “messenger”—is tumbling chaotically as the result of a violent collision.
And the interstellar object will continue to spin for billions of years as it journeys through space, scientists have reported in a study publis

Example of a neutral paragraph:
'Oumuamua: Space Cigar Is Still Spinning From Mysterious Violent Collision


NOTE: Paragraph splitting via '\n\n' is a heuristic approximation. Some "paragraphs" may be titles or single sentences rather than full argumentative units. A more sophisticated sentence-level splitting could be applied in future iterations.

In [12]:
# Saving
df_en.to_csv(f'{BASE}/data/processed/ptc_en.csv', index=False)

print(f"Saved: {len(df_en)} paragraphs")
print(f"File: {BASE}/data/processed/ptc_en.csv")

Saved: 1147 paragraphs
File: /content/drive/MyDrive/NU/SEDS/multilingual-persuasion-detection/data/processed/ptc_en.csv


## baiangali/fake_news Dataset (Kazakh + Russian — proxy task)

In [13]:
kz_dir = f'{BASE}/data/raw/baiangali'
os.makedirs(kz_dir, exist_ok=True)
os.chdir(kz_dir)

!git clone https://github.com/baiangali/fake_news.git

os.chdir('/content')
print(os.listdir(f'{kz_dir}/fake_news'))

Cloning into 'fake_news'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 13 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 131.12 KiB | 7.28 MiB/s, done.
['.git', 'README.md', 'fake_real_news_annotated.json']


In [17]:
json_path = f'{kz_dir}/fake_news/fake_real_news_annotated.json'

with open(json_path, 'r', encoding='utf-8') as f:
     raw = json.load(f)

print(f"Data type: {type(raw)}")
print(f"Number of elements: {len(raw)}")
print("\nFirst element:")
print(json.dumps(raw[0], ensure_ascii=False, indent=2)[:1000])

Data type: <class 'list'>
Number of elements: 204

First element:
{
  "id": 411,
  "annotations": [
    {
      "id": 411,
      "completed_by": 1,
      "result": [
        {
          "from_name": "label",
          "to_name": "text",
          "type": "labels",
          "value": {
            "start": 0,
            "end": 33,
            "text": "2025 жылғы 8–9 маусымда",
            "labels": [
              "TIME"
            ]
          }
        },
        {
          "from_name": "label",
          "to_name": "text",
          "type": "labels",
          "value": {
            "start": 34,
            "end": 123,
            "text": "Қазақстан Президенті Қасым-Жомарт Тоқаевтың шақыруымен Болгария Президенті Румен Радев",
            "labels": [
              "SOURCE"
            ]
          }
        },
        {
          "from_name": "label",
          "to_name": "text",
          "type": "labels",
          "value": {
            "start": 124,
            "end": 162,
     

Great, the structure is clear. Now we need to find where exactly the main_classification is stored, that is, where the article text is and where the REAL_NEWS/FAKE_NEWS tag is.

In [18]:
# Look at ALL from_name values ​​in the first element
print("All 'from_name' in the first element:")
for ann in raw[0]['annotations']:
    for result in ann['result']:
        print(f"  from_name: {result.get('from_name')}")
        print(f"  value keys: {list(result.get('value', {}).keys())}")
        print()

# And look at the 'data' field - there should be text there
print("Поле 'data':")
print(json.dumps(raw[0].get('data', {}), ensure_ascii=False, indent=2))

All 'from_name' in the first element:
  from_name: label
  value keys: ['start', 'end', 'text', 'labels']

  from_name: label
  value keys: ['start', 'end', 'text', 'labels']

  from_name: label
  value keys: ['start', 'end', 'text', 'labels']

  from_name: label
  value keys: ['start', 'end', 'text', 'labels']

  from_name: label
  value keys: ['start', 'end', 'text', 'labels']

  from_name: main_classification
  value keys: ['choices']

  from_name: author_intent_subtype
  value keys: ['choices']

  from_name: target_audience_subtype
  value keys: ['choices']

Поле 'data':
{
  "text": "2025 жылғы 8–9 маусымда Қазақстан Президенті Қасым-Жомарт Тоқаевтың шақыруымен Болгария Президенті Румен Радев Астанаға ресми сапармен келеді. Жоғары деңгейдегі келіссөздер барысында қазақ-болгар сауда-экономикалық және инвестициялық ынтымақтастығының перспективалары талқыланбақ. Президенттер кездесуі екі елдің байланысын жаңа деңгейге көтеруді көздейді."
}


Now everything is clear. Text is in data.text, label is in from_name: main_classification → value.choices[0]. Language detection using langdetect.

In [22]:
def detect_lang(text):
    try:
        code = detect(str(text))
        return 'kz' if code == 'kk' else code
    except LangDetectException:
        return 'unknown'

records_baiangali = []

for item in raw:
    # Text
    text = item.get('data', {}).get('text', '')
    if not text:
        continue

    # Looking for main_classification
    label = None
    for ann in item.get('annotations', []):
        for result in ann.get('result', []):
            if result.get('from_name') == 'main_classification':
                choices = result.get('value', {}).get('choices', [])
                if choices:
                    label = choices[0]
                    break

    if label is None:
        continue

    # Language detection
    lang = detect_lang(text)

    records_baiangali.append({
        'text': text,
        'original_label': label,
        'has_persuasion': 1 if label == 'FAKE_NEWS' else 0,
        'language': lang,
        'source': 'baiangali'
    })

df_baiangali = pd.DataFrame(records_baiangali)

print(f"Total loaded: {len(df_baiangali)}")
print("\Languages:")
print(df_baiangali['language'].value_counts())
print("\nМетки:")
print(df_baiangali['original_label'].value_counts())
print("\Language × Label:")
print(df_baiangali.groupby(['language', 'original_label']).size())

<>:43: SyntaxWarning: invalid escape sequence '\L'
<>:47: SyntaxWarning: invalid escape sequence '\L'
<>:43: SyntaxWarning: invalid escape sequence '\L'
<>:47: SyntaxWarning: invalid escape sequence '\L'
/tmp/ipykernel_393/677946397.py:43: SyntaxWarning: invalid escape sequence '\L'
  print("\Languages:")
/tmp/ipykernel_393/677946397.py:47: SyntaxWarning: invalid escape sequence '\L'
  print("\Language × Label:")


Total loaded: 204
\Languages:
language
ru    184
uk     20
Name: count, dtype: int64

Метки:
original_label
FAKE_NEWS    104
REAL_NEWS    100
Name: count, dtype: int64
\Language × Label:
language  original_label
ru        FAKE_NEWS         91
          REAL_NEWS         93
uk        FAKE_NEWS         13
          REAL_NEWS          7
dtype: int64


langdetect didn't recognize Kazakh—it identified everything as Russian (ru) and Ukrainian (uk). This is a known weakness of langdetect: Kazakh and Russian use the Cyrillic alphabet and are very similar phonetically, so langdetect confuses them.

Let's check manually: the first element was clearly Kazakh (it contained Kazakh words like "жылғы" and "Президенті"). Let's look at some texts:

In [23]:
# Looking at the first 5 texts with a specific language
print("The first 3 texts are identified as 'ru':")
for text in df_baiangali[df_baiangali['language']=='ru']['text'].head(3):
    print(f"  {text[:150]}")
    print()

print("The first 3 texts are identified as 'uk':")
for text in df_baiangali[df_baiangali['language']=='uk']['text'].head(3):
    print(f"  {text[:150]}")
    print()

The first 3 texts are identified as 'ru':
  2025 жылғы 8–9 маусымда Қазақстан Президенті Қасым-Жомарт Тоқаевтың шақыруымен Болгария Президенті Румен Радев Астанаға ресми сапармен келеді. Жоғары 

  2023 жылғы 19 наурызда Қазақстанда Мәжіліс пен мәслихаттар сайлауы өтіп, нәтижесінде 6 партия Парламентке өтті. Ресми қорытынды бойынша «Amanat» парти

  Үкімет 2024 жылы әскерге шақырылатын жастардың санын арттыру туралы қаулы жобасын әзірледі. Құжатқа сәйкес, 2024 жылы көктемгі және күзгі шақырылымдар

The first 3 texts are identified as 'uk':
  Қазақстан өз мұнайын экспорттауда балама бағыттарды игере бастады. 2023 жылы Каспий теңізі арқылы Әзербайжанның Баку портына қазақ мұнайын танкермен ж

  20 жастағы қазақ шахматшысы Бибісара Асаубаева әйелдер арасындағы блиц шахматтан екі дүркін әлем чемпионы (2021, 2022) атанды. 2023 жылдың желтоқсанын

  2023 жылғы наурызда футболдан Қазақстан ұлттық құрамасы 2024 Еуропа чемпионатының іріктеуінде тарихи жеңіске қол жеткізді. Астана қаласында өтке

Let's make a detector manually

In [24]:
# Kazakh-specific characters not found in Russian
KAZAKH_CHARS = set('әғқңөұүһі')

def detect_kz_ru(text):
    """
    Detects Kazakh vs Russian based on unique Kazakh characters.
    Kazakh uses Cyrillic but has 9 unique letters absent in Russian.
    """
    text_lower = text.lower()
    kazakh_char_count = sum(1 for ch in text_lower if ch in KAZAKH_CHARS)
    # If text contains at least 2 unique Kazakh characters → Kazakh
    return 'kz' if kazakh_char_count >= 2 else 'ru'

# Apply to all texts
df_baiangali['language'] = df_baiangali['text'].apply(detect_kz_ru)

print("Language distribution after fix:")
print(df_baiangali['language'].value_counts())
print("\nLanguage × Label:")
print(df_baiangali.groupby(['language', 'original_label']).size())

# Verify: check first 3 texts per language
print("\nFirst KZ text sample:")
print(df_baiangali[df_baiangali['language']=='kz'].iloc[0]['text'][:200])
print("\nFirst RU text sample:")
print(df_baiangali[df_baiangali['language']=='ru'].iloc[0]['text'][:200])

Language distribution after fix:
language
ru    103
kz    101
Name: count, dtype: int64

Language × Label:
language  original_label
kz        FAKE_NEWS         50
          REAL_NEWS         51
ru        FAKE_NEWS         54
          REAL_NEWS         49
dtype: int64

First KZ text sample:
2025 жылғы 8–9 маусымда Қазақстан Президенті Қасым-Жомарт Тоқаевтың шақыруымен Болгария Президенті Румен Радев Астанаға ресми сапармен келеді. Жоғары деңгейдегі келіссөздер барысында қазақ-болгар сауд

First RU text sample:
В соцсетях активно распространяется ролик, где глава Липецкой области Игорь Артамонов якобы объявляет о введении с 1 июля единого военного сбора в размере 35% от доходов граждан. Утверждается, что так


Excellent! Now everything is correctly identified:
- 101 Kazakh texts (50 fake / 51 real)
- 103 Russian texts (54 fake / 49 real)
- The class balance is almost perfect in both languages

In [25]:
# Split by language and save
df_kz = df_baiangali[df_baiangali['language'] == 'kz'][
    ['text', 'has_persuasion', 'language', 'source']].copy()

df_ru = df_baiangali[df_baiangali['language'] == 'ru'][
    ['text', 'has_persuasion', 'language', 'source']].copy()

df_kz.to_csv(f'{BASE}/data/processed/baiangali_kz.csv', index=False)
df_ru.to_csv(f'{BASE}/data/processed/baiangali_ru.csv', index=False)

print(f"Saved:")
print(f"  KZ: {len(df_kz)} texts → baiangali_kz.csv")
print(f"  RU: {len(df_ru)} texts → baiangali_ru.csv")

Saved:
  KZ: 101 texts → baiangali_kz.csv
  RU: 103 texts → baiangali_ru.csv


## Key Design Decisions
- PTC articles are split into **paragraphs** and labeled based on
  character-level span overlap with annotated persuasion spans.
  This transforms the article-level task into paragraph-level
  classification and improves class balance (56% / 44%).
- baiangali labels (FAKE_NEWS / REAL_NEWS) are mapped to binary
  `has_persuasion` (1 / 0) as a proxy for manipulative content.
- Language detection for baiangali uses a keyword-based heuristic
  (Kazakh-specific characters) rather than langdetect, which
  incorrectly classifies Kazakh as Russian due to shared Cyrillic script.

## Summary: Data Loading Complete

| Dataset | Language | Task | Texts | Balance |
|---------|----------|------|-------|---------|
| PTC SemEval-2020 | EN | Persuasion techniques (paragraphs) | 1147 | 56% / 44% |
| baiangali/fake_news | KZ | Fake/real news (proxy) | 101 | 50% / 50% |
| baiangali/fake_news | RU | Fake/real news (proxy) | 103 | 52% / 48% |

**Note on proxy datasets:** KZ and RU datasets are used as test-only
proxy tasks. No Kazakh or Russian persuasion-annotated corpus exists
in open access — this is documented as Finding 3 (Resource Gap).

**Access barriers:** SemEval-2023 Task 3 (Detection of
Persuasion Techniques in Political News Articles) includes Russian
and would have been the ideal Russian-language dataset for this study.
However, registration requires institutional affiliation verification,
which was not available at the time of this project. This further
confirms the resource gap for Russian-language persuasion research
in non-institutional contexts.

## Outputs
| File | Description |
|------|-------------|
| `data/processed/ptc_en.csv` | 1147 EN paragraphs with persuasion labels |
| `data/processed/baiangali_kz.csv` | 101 KZ texts with fake/real labels |
| `data/processed/baiangali_ru.csv` | 103 RU texts with fake/real labels |



**Next:** `02_eda.ipynb` — Exploratory Data Analysis